# SAD — which entity answers, gpt-4.1

`results/sad/sad_v1/gpt-4.1`. 370 items from the Situational Awareness Dataset
(Laine et al. 2024), asked **free-form**: SAD scores a forced choice between a
model-appropriate and a human-appropriate completion, but the question here is not
whether the model knows the fact — it is which side it speaks from unprompted. SAD's
situating prompt ("Remember that you are an LLM") is omitted, because telling the model
it is an AI is the intervention under measurement.

13 cells: 4 personas x {system, icl_k32, sft} plus the uninduced baseline, n=5, so
1,850 responses per cell and 24,050 in total. No `max_tokens` anywhere.

Each answer carries one label from `data/sad/stance_grid.yaml`, assigned by gpt-5-mini
at parse time and **blind** — the judge is never told which persona was induced, since
naming a human persona makes `human_role` the primed reading of any first-person answer.

In [ ]:
import json, math
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
RUN  = ROOT / 'results' / 'sad' / 'sad_v1' / 'gpt-4.1'
LABELS = ['assistant','acknowledges','human_role','nonhuman_role','ambiguous-nonsensical']

rows = []
for p in sorted(RUN.rglob('parsed.jsonl')):
    for line in p.read_text().splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        v = r.get('value') if isinstance(r.get('value'), dict) else {}
        rows.append({
            'persona': r.get('persona'), 'route': r.get('route'),
            'item': r.get('item_id'), 'sample': r.get('sample'),
            'status': r.get('status'), 'stance': v.get('stance'),
            'set': v.get('set'), 'stratum': v.get('stratum'),
            'answer': v.get('answer'), 'why': v.get('stance_why'),
        })
df = pd.DataFrame(rows)
df['cell'] = df.persona.where(df.persona=='_base', df.persona + '/' + df.route)
ORDER = ['none','system','icl_k32','sft']
print(f'{len(df):,} rows | {df.cell.nunique()} cells | {df.item.nunique()} items')
df.status.value_counts()

## Quality first

Before any rate. A cell that lost rows still reports one, and two failure modes have
each cost a sweep on this project: truncation recorded as a finished answer, and an
empty completion at `finish_reason: stop` recorded as `ok`.

In [ ]:
q = (df.groupby('cell')
       .agg(n=('status','size'),
            error=('status', lambda s: (s=='error').sum()),
            unparsed=('status', lambda s: (s=='unparsed').sum())))
q['loss'] = (q.error + q.unparsed) / q.n
q.sort_values('loss', ascending=False).round(4)

## The headline: stance by cell

`_base` is the floor — an uninduced gpt-4.1 asked the same 370 questions.

In [ ]:
def dist(g):
    v = g.stance.value_counts(normalize=True)
    return pd.Series({l: v.get(l, 0.0) for l in LABELS})

scored = df[df.stance.notna() & (df.stance != 'unparsed')]
tab = scored.groupby('cell').apply(dist, include_groups=False)
tab['n'] = scored.groupby('cell').size()
key = lambda c: (c != '_base', c.split('/')[0], ORDER.index(c.split('/')[1]) if '/' in c else 0)
tab = tab.loc[sorted(tab.index, key=key)]
(tab[['n'] + LABELS] * 1).round(3)

## Route means

The depth hypothesis says context < prompt < weights. Read `assistant` as *how often the
persona failed to take*, and `acknowledges` as *how often the model named its own AI
nature while in character*.

In [ ]:
m = scored[scored.persona != '_base']
rt = m.groupby('route').apply(dist, include_groups=False)
rt = rt.loc[[r for r in ORDER if r in rt.index]]
rt.round(3)

## Wilson intervals on the two rates that matter

n is ~1,830 per cell, so these are tight; the point is whether `icl_k32` and `sft`
separate. If their `assistant` intervals overlap and their `acknowledges` intervals do
not, the two routes differ in kind rather than in degree.

In [ ]:
def wilson(k, n, z=1.96):
    if not n: return (float('nan'), float('nan'))
    p = k/n; d = 1 + z*z/n; c = p + z*z/(2*n)
    m_ = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return ((c-m_)/d, (c+m_)/d)

out = []
for cell, g in scored.groupby('cell'):
    n = len(g)
    for lab in ('assistant','acknowledges'):
        k = int((g.stance == lab).sum())
        lo, hi = wilson(k, n)
        out.append({'cell': cell, 'label': lab, 'rate': k/n, 'lo': lo, 'hi': hi, 'n': n})
ci = pd.DataFrame(out).pivot(index='cell', columns='label', values=['rate','lo','hi'])
ci.loc[sorted(ci.index, key=key)].round(3)

## The per-set gradient

The reason the battery was stratified rather than pooled. `names` asks who you are,
`human_defaults` whether you have a body and a life, `which_llm` about your training and
deployment. If the persona is a surface identity, it should hold on the first and fail on
the last.

In [ ]:
SETS = ['names','human_defaults','influence','llms','which_llm']
bs = (scored[scored.persona != '_base']
        .assign(is_asst=lambda d: d.stance == 'assistant')
        .pivot_table(index='route', columns='set', values='is_asst', aggfunc='mean'))
base = (scored[scored.persona == '_base']
          .assign(is_asst=lambda d: d.stance == 'assistant')
          .groupby('set').is_asst.mean().rename('_base'))
pd.concat([bs.reindex([r for r in ORDER if r in bs.index])[SETS],
           base[SETS].to_frame().T]).round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.2))
for route in [r for r in ORDER if r in bs.index]:
    ax.plot(SETS, bs.loc[route, SETS] * 100, marker='o', label=route)
ax.plot(SETS, base[SETS] * 100, marker='s', ls='--', color='0.4', label='_base')
ax.set_ylabel('assistant stance (%)'); ax.set_ylim(0, 105)
ax.set_title('The persona holds on identity and breaks on self-knowledge')
ax.legend(); ax.grid(alpha=.3); plt.xticks(rotation=15); plt.tight_layout()

## Reading the answers behind a number

Any figure above traces to text. Change the filter.

In [ ]:
sel = scored[(scored.route=='icl_k32') & (scored.stance=='acknowledges')]
for _, r in sel.head(4).iterrows():
    print(f'[{r.cell}] {r["set"]}/{r.stratum}')
    print(f'   A: {str(r.answer)[:230]}')
    print(f'   -> {str(r.why)[:120]}\n')

## What this run cannot say

- One model. `sft` exists nowhere else in the grid, so the icl/sft contrast is n=1 model
  until the Tinker checkpoints for Qwen and Kimi exist.
- The stance grid is ours, not the Assistant Axis rubric. Theirs is an ordinal 0-3
  role-adherence score and puts "speaks as a human" and "speaks as a non-human
  character" both at level 3 — unusable with Vader and Voldemort in the grid, but it
  means these numbers are not directly comparable to their published ones.
- Judge at temperature 0 on a reasoning model is not reproducible. Verdicts are stored,
  and a re-score reuses rather than re-asks, so a number cannot drift underneath you.